# VGen: Benchmarking LLMs for Automated Verilog RTL Code Generation
## LLM4ChipDesign Undergraduate Course Tutorial

Welcome to this comprehensive tutorial on using Large Language Models (LLMs) for automated Verilog hardware description language generation!

### Learning Objectives
By the end of this tutorial, you will:
- Understand how LLMs can generate Verilog RTL code
- Learn to use fine-tuned models for hardware design
- Evaluate generated Verilog code for syntax and functional correctness
- Explore different complexity levels of hardware designs
- Compare different LLM approaches for Verilog generation

### Course Context
This notebook is part of the **LLM4ChipDesign** course, exploring the intersection of:
- 🤖 Large Language Models (transformers, code generation)
- 💻 Hardware Description Languages (Verilog, RTL design)
- 🔧 Digital Design (combinational/sequential circuits, FSMs)
- ✅ Verification (testbenches, simulation)

### Research Background
Based on the paper: **"Benchmarking Large Language Models for Automated Verilog RTL Code Generation"** by Thakur et al.
- 📄 [arXiv Link](https://arxiv.org/abs/2212.11140)
- 🏛️ NYU Tandon School of Engineering

### Prerequisites
- Basic understanding of digital logic design
- Familiarity with Verilog syntax (helpful but not required)
- Python programming basics
- Understanding of LLM concepts (transformers, tokenization)

## 1. Environment Setup

First, let's set up our environment with all necessary dependencies.

In [ ]:
# Install required packages
!pip install torch transformers accelerate sentencepiece --quiet

print("✓ PyTorch and Transformers installed successfully!")

# Check for GPU availability
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"  Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("⚠ No GPU detected - using CPU (this will be slower)")
    print("  For better performance, enable GPU in Runtime > Change runtime type > Hardware accelerator")

In [ ]:
# Clone the VGen repository
import os

if not os.path.exists('VGen'):
    !git clone https://github.com/shailja-thakur/VGen.git
    print("✓ VGen repository cloned successfully!")
else:
    print("✓ VGen repository already exists!")

%cd VGen

## 2. Understanding VGen: Theory and Concepts

### What is VGen?

VGen is a benchmark framework for evaluating Large Language Models on **Verilog RTL code generation**. It includes:
1. **Fine-tuned models** trained on Verilog datasets from GitHub and textbooks
2. **Benchmark problems** of varying complexity (basic, intermediate, advanced)
3. **Evaluation framework** with testbenches for functional verification

### Why Use LLMs for Hardware Design?

#### Traditional Hardware Design Challenges:
- ⏰ Time-consuming manual RTL coding
- 🐛 Human error in implementation
- 📚 Steep learning curve for HDLs
- 🔄 Repetitive patterns in design

#### LLM Advantages:
- ⚡ Rapid prototyping from natural language descriptions
- 🎯 Pattern recognition from vast code repositories
- 🤝 Assistance for novice designers
- 🔧 Automated generation of common modules

### The VGen Approach

```
Natural Language Prompt → Fine-tuned LLM → Verilog Code → Testbench → Verification
```

### Model Architecture: CodeGen

VGen uses **CodeGen**, a family of autoregressive language models:
- Based on GPT-style transformer architecture
- Pre-trained on code from multiple programming languages
- **Fine-tuned specifically on Verilog datasets**
- Available in multiple sizes (350M, 2B, 6B, 16B parameters)

### Benchmark Hierarchy

| Level | Complexity | Examples |
|-------|-----------|----------|
| **Basic** | Simple combinational logic | Wire, AND gate, MUX, Priority encoder |
| **Intermediate** | Sequential logic, FSMs | Half adder, Counter, LFSR, Moore FSM, RAM |
| **Advanced** | Complex state machines, arithmetic | Signed addition, FSM recognizers, Arithmetic shifter |

### Evaluation Metrics

1. **Syntax Correctness**: Does the code compile?
2. **Functional Correctness**: Does it pass testbenches?
3. **Code Quality**: Efficiency, readability, best practices

## 3. Exploring the Benchmark Suite

Let's examine the structure of VGen's benchmark problems.

In [ ]:
import os
import glob
from collections import defaultdict

# Directory containing all benchmarks
benchmarks_dir = 'prompts-and-testbenches'

# Categorize benchmarks
categories = defaultdict(list)

if os.path.exists(benchmarks_dir):
    for item in sorted(os.listdir(benchmarks_dir)):
        if os.path.isdir(os.path.join(benchmarks_dir, item)):
            if item.startswith('basic'):
                categories['Basic'].append(item)
            elif item.startswith('intermediate'):
                categories['Intermediate'].append(item)
            elif item.startswith('advanced'):
                categories['Advanced'].append(item)
    
    print("VGen Benchmark Suite Overview")
    print("=" * 70)
    
    for category, benchmarks in categories.items():
        print(f"\n📁 {category} Level ({len(benchmarks)} problems)")
        print("-" * 70)
        for bench in benchmarks:
            # Try to find description from files
            bench_path = os.path.join(benchmarks_dir, bench)
            files = os.listdir(bench_path)
            print(f"  • {bench:20s} - {len(files)} files")
    
    print(f"\n📊 Total Benchmarks: {sum(len(v) for v in categories.values())}")
else:
    print("Benchmarks directory not found. Make sure you're in the VGen directory.")

In [ ]:
# Examine the structure of a benchmark problem
sample_benchmark = 'basic1'  # wire assignment
sample_path = os.path.join(benchmarks_dir, sample_benchmark)

if os.path.exists(sample_path):
    print(f"Structure of '{sample_benchmark}' benchmark:")
    print("=" * 70)
    
    files = sorted(os.listdir(sample_path))
    
    file_categories = {
        'Prompts': [],
        'Answer': [],
        'Testbench': [],
        'Other': []
    }
    
    for f in files:
        if f.startswith('prompt'):
            file_categories['Prompts'].append(f)
        elif f.startswith('answer'):
            file_categories['Answer'].append(f)
        elif f.startswith('tb_'):
            file_categories['Testbench'].append(f)
        else:
            file_categories['Other'].append(f)
    
    for category, items in file_categories.items():
        if items:
            print(f"\n{category}:")
            for item in items:
                print(f"  📄 {item}")
    
    print("\n" + "=" * 70)
    print("Key Components:")
    print("  • Prompts: Different phrasings of the same design specification")
    print("  • Answer: Reference implementation (correct Verilog code)")
    print("  • Testbench: Verification code to check functional correctness")

In [ ]:
# Display example prompt and answer
def display_file(filepath, title):
    """Display file contents with formatting"""
    if os.path.exists(filepath):
        print(f"\n{title}")
        print("=" * 70)
        with open(filepath, 'r') as f:
            content = f.read()
        print(content)
        print("=" * 70)

# Display wire assignment example
display_file(f'{benchmarks_dir}/basic1/prompt1_wire_assign.v', 
             '📝 Example Prompt: Wire Assignment')

display_file(f'{benchmarks_dir}/basic1/answer_wire_assign.v', 
             '✅ Reference Answer: Wire Assignment')

## 4. Loading the Fine-tuned Verilog Model

Now we'll load a fine-tuned CodeGen model specifically trained on Verilog code.

### Available Models on HuggingFace:

| Model Name | Parameters | Description |
|------------|-----------|-------------|
| `shailja/fine-tuned-codegen-2B-Verilog` | 2B | Recommended for Colab |
| `shailja/fine-tuned-codegen-6B-Verilog` | 6B | Requires more GPU memory |
| `shailja/fine-tuned-codegen-16B-Verilog` | 16B | Requires high-end GPU |

We'll use the **2B model** which balances quality and computational requirements.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading VGen fine-tuned model...")
print("This may take a few minutes on first run (downloading ~8GB model)\n")

# Model configuration
model_name = "shailja/fine-tuned-codegen-2B-Verilog"

try:
    # Load tokenizer
    print("[1/2] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("      ✓ Tokenizer loaded")
    
    # Load model
    print("\n[2/2] Loading model (this takes longer)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True
    )
    
    # Move to device
    model = model.to(device)
    model.eval()  # Set to evaluation mode
    
    print("      ✓ Model loaded and ready!")
    print(f"\n📊 Model Statistics:")
    print(f"   • Parameters: ~2 billion")
    print(f"   • Device: {device}")
    print(f"   • Precision: {next(model.parameters()).dtype}")
    
    MODEL_LOADED = True
    
except Exception as e:
    print(f"\n❌ Error loading model: {e}")
    print("\nTroubleshooting:")
    print("  • Check internet connection")
    print("  • Verify GPU is enabled (Runtime > Change runtime type)")
    print("  • Try restarting the runtime")
    MODEL_LOADED = False

## 5. Verilog Code Generation Functions

Let's create utility functions for generating Verilog code with the LLM.

In [ ]:
def generate_verilog(prompt, 
                     max_length=256,
                     temperature=0.2,
                     top_p=0.95,
                     num_return_sequences=1,
                     stop_token="endmodule"):
    """
    Generate Verilog code from a natural language or partial code prompt.
    
    Args:
        prompt: Input prompt (comment or partial code)
        max_length: Maximum tokens to generate
        temperature: Sampling temperature (lower = more deterministic)
        top_p: Nucleus sampling parameter
        num_return_sequences: Number of different outputs to generate
        stop_token: Token to stop generation
    
    Returns:
        List of generated Verilog code strings
    """
    if not MODEL_LOADED:
        print("❌ Model not loaded. Please run the model loading cell first.")
        return []
    
    # Tokenize input
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode outputs
    results = []
    for output in outputs:
        generated_text = tokenizer.decode(output, skip_special_tokens=True)
        
        # Add endmodule if not present
        if stop_token in generated_text:
            generated_text = generated_text.split(stop_token)[0] + stop_token
        elif not generated_text.strip().endswith('endmodule'):
            generated_text += "\nendmodule"
        
        results.append(generated_text)
    
    return results


def display_generated_code(code, title="Generated Verilog Code"):
    """Display generated code with syntax highlighting"""
    print(f"\n{title}")
    print("=" * 70)
    print(code)
    print("=" * 70)


def analyze_code(code):
    """Basic analysis of generated Verilog code"""
    lines = code.split('\n')
    
    stats = {
        'total_lines': len(lines),
        'code_lines': len([l for l in lines if l.strip() and not l.strip().startswith('//')]),
        'comment_lines': len([l for l in lines if l.strip().startswith('//')]),
        'has_module': 'module' in code,
        'has_endmodule': 'endmodule' in code,
        'has_always': 'always' in code,
        'has_assign': 'assign' in code,
    }
    
    print("\n📊 Code Analysis:")
    print("-" * 70)
    print(f"  Total lines:        {stats['total_lines']}")
    print(f"  Code lines:         {stats['code_lines']}")
    print(f"  Comment lines:      {stats['comment_lines']}")
    print(f"  Has module/endmodule: {'✓' if stats['has_module'] and stats['has_endmodule'] else '✗'}")
    print(f"  Uses 'always' blocks: {'✓' if stats['has_always'] else '✗'}")
    print(f"  Uses 'assign':        {'✓' if stats['has_assign'] else '✗'}")
    
    return stats

print("✓ Generation utilities defined!")

## 6. Basic Examples: Learning the Fundamentals

Let's start with basic combinational logic examples to understand how the model works.

In [ ]:
# Example 1: Simple Wire Assignment
print("🔹 EXAMPLE 1: Wire Assignment")
print("="*70)

prompt_wire = """// Create a module with one input and one output that behaves like a wire
module wire_assign( input in, output out );"""

print("\n📝 Prompt:")
print(prompt_wire)

if MODEL_LOADED:
    generated = generate_verilog(prompt_wire, max_length=100, temperature=0.2)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
    
    # Show reference answer for comparison
    with open(f'{benchmarks_dir}/basic1/answer_wire_assign.v', 'r') as f:
        reference = f.read()
    display_generated_code(reference, "✅ Reference Answer")
else:
    print("\n⚠ Model not loaded - skipping generation")

In [ ]:
# Example 2: AND Gate
print("🔹 EXAMPLE 2: AND Gate")
print("="*70)

prompt_and = """// Implement a 2-input AND gate
module and_gate(input a, input b, output out);"""

print("\n📝 Prompt:")
print(prompt_and)

if MODEL_LOADED:
    generated = generate_verilog(prompt_and, max_length=100, temperature=0.2)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
else:
    print("\n⚠ Model not loaded - skipping generation")

In [ ]:
# Example 3: 2-to-1 Multiplexer
print("🔹 EXAMPLE 3: 2-to-1 Multiplexer")
print("="*70)

prompt_mux = """// Create a 2-to-1 multiplexer
// When sel=0, output = a; when sel=1, output = b
module mux2to1(input a, input b, input sel, output out);"""

print("\n📝 Prompt:")
print(prompt_mux)

if MODEL_LOADED:
    # Generate multiple versions to show diversity
    generated = generate_verilog(prompt_mux, max_length=150, 
                                temperature=0.5, num_return_sequences=2)
    
    for i, code in enumerate(generated, 1):
        display_generated_code(code, f"🤖 Model Generated Code (Version {i})")
        analyze_code(code)
        print()
else:
    print("\n⚠ Model not loaded - skipping generation")

## 7. Intermediate Examples: Sequential Logic

Now let's explore more complex sequential circuits including counters and state machines.

In [ ]:
# Example 4: Half Adder
print("🔹 EXAMPLE 4: Half Adder")
print("="*70)

prompt_ha = """// Create a half adder circuit
// Outputs: sum and carry
module half_adder(input a, input b, output sum, output cout);"""

print("\n📝 Prompt:")
print(prompt_ha)

if MODEL_LOADED:
    generated = generate_verilog(prompt_ha, max_length=150, temperature=0.2)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
else:
    print("\n⚠ Model not loaded - skipping generation")

In [ ]:
# Example 5: Counter (1 to 12)
print("🔹 EXAMPLE 5: Counter (1 to 12)")
print("="*70)

prompt_counter = """// Design a counter that counts from 1 to 12
// Reset to 1 on reset signal
// Wrap around to 1 after reaching 12
module counter_1to12(
    input clk,
    input reset,
    input enable,
    output reg [3:0] count
);"""

print("\n📝 Prompt:")
print(prompt_counter)

if MODEL_LOADED:
    generated = generate_verilog(prompt_counter, max_length=250, temperature=0.3)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
else:
    print("\n⚠ Model not loaded - skipping generation")

In [ ]:
# Example 6: Simple Moore FSM
print("🔹 EXAMPLE 6: Moore Finite State Machine")
print("="*70)

prompt_fsm = """// Create a simple 2-state Moore FSM
// States: S0 and S1
// Transition from S0 to S1 when input is 1
// Transition from S1 to S0 when input is 0
// Output is 1 in state S1, 0 in state S0
module moore_fsm(
    input clk,
    input reset,
    input in,
    output reg out
);"""

print("\n📝 Prompt:")
print(prompt_fsm)

if MODEL_LOADED:
    generated = generate_verilog(prompt_fsm, max_length=300, temperature=0.3)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
else:
    print("\n⚠ Model not loaded - skipping generation")

## 8. Advanced Examples: Complex Designs

Let's tackle more sophisticated designs that test the limits of LLM code generation.

In [ ]:
# Example 7: Signed Addition with Overflow Detection
print("🔹 EXAMPLE 7: Signed Addition with Overflow")
print("="*70)

prompt_signed = """// Implement a signed 8-bit adder with overflow detection
// Detect overflow when adding two signed numbers
module signed_adder(
    input signed [7:0] a,
    input signed [7:0] b,
    output signed [7:0] sum,
    output overflow
);"""

print("\n📝 Prompt:")
print(prompt_signed)

if MODEL_LOADED:
    generated = generate_verilog(prompt_signed, max_length=250, temperature=0.3)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
else:
    print("\n⚠ Model not loaded - skipping generation")

In [ ]:
# Example 8: Sequence Detector FSM
print("🔹 EXAMPLE 8: Sequence Detector (101)")
print("="*70)

prompt_detector = """// Design an FSM that detects the sequence "101" in a serial input stream
// Output should be 1 when the sequence is detected
// Overlapping sequences should be detected
module sequence_detector_101(
    input clk,
    input reset,
    input in,
    output reg detected
);"""

print("\n📝 Prompt:")
print(prompt_detector)

if MODEL_LOADED:
    generated = generate_verilog(prompt_detector, max_length=400, temperature=0.3)
    display_generated_code(generated[0], "🤖 Model Generated Code")
    analyze_code(generated[0])
else:
    print("\n⚠ Model not loaded - skipping generation")

## 9. Exploring Generation Parameters

Let's understand how different parameters affect code generation quality and diversity.

In [ ]:
# Compare different temperatures
print("🔬 PARAMETER EXPLORATION: Temperature Effect")
print("="*70)

test_prompt = """// Create a simple XOR gate
module xor_gate(input a, input b, output out);"""

temperatures = [0.1, 0.5, 1.0]

if MODEL_LOADED:
    for temp in temperatures:
        print(f"\n🌡️  Temperature = {temp}")
        print("-" * 70)
        
        # Generate 2 samples
        generated = generate_verilog(test_prompt, max_length=120, 
                                    temperature=temp, num_return_sequences=2)
        
        for i, code in enumerate(generated, 1):
            print(f"\nSample {i}:")
            print(code)
            print()
    
    print("\n📊 Analysis:")
    print("  • Low temperature (0.1):  More deterministic, focused on most likely tokens")
    print("  • Medium temperature (0.5): Balanced creativity and correctness")
    print("  • High temperature (1.0):  More diverse but potentially less correct")
else:
    print("\n⚠ Model not loaded - skipping generation")

## 10. Code Verification Framework

Let's create tools to verify the generated Verilog code.

In [ ]:
import re

class VerilogVerifier:
    """Simple Verilog code verification tools"""
    
    @staticmethod
    def check_syntax_patterns(code):
        """
        Check for common syntax patterns (not a full syntax checker)
        """
        issues = []
        
        # Check for module declaration
        if not re.search(r'module\s+\w+', code):
            issues.append("Missing module declaration")
        
        # Check for endmodule
        if 'endmodule' not in code:
            issues.append("Missing 'endmodule'")
        
        # Check for balanced parentheses
        if code.count('(') != code.count(')'):
            issues.append("Unbalanced parentheses")
        
        # Check for balanced begin/end
        begin_count = len(re.findall(r'\bbegin\b', code))
        end_count = len(re.findall(r'\bend\b', code))
        if begin_count != end_count:
            issues.append(f"Unbalanced begin/end blocks ({begin_count} begin, {end_count} end)")
        
        # Check for semicolons in assignments
        assign_lines = re.findall(r'assign.*', code)
        for line in assign_lines:
            if not line.strip().endswith(';'):
                issues.append(f"Missing semicolon in assign: {line.strip()[:50]}...")
        
        return issues
    
    @staticmethod
    def extract_module_info(code):
        """
        Extract basic module information
        """
        info = {}
        
        # Extract module name
        module_match = re.search(r'module\s+(\w+)', code)
        if module_match:
            info['module_name'] = module_match.group(1)
        
        # Count inputs
        info['num_inputs'] = len(re.findall(r'input\s+', code))
        
        # Count outputs
        info['num_outputs'] = len(re.findall(r'output\s+', code))
        
        # Check for registers
        info['has_registers'] = 'reg' in code
        
        # Check for always blocks
        info['num_always_blocks'] = len(re.findall(r'always\s*@', code))
        
        # Check for assign statements
        info['num_assigns'] = len(re.findall(r'assign\s+', code))
        
        return info
    
    @staticmethod
    def verify_code(code):
        """
        Run all verification checks
        """
        print("\n🔍 VERIFICATION REPORT")
        print("="*70)
        
        # Syntax pattern checks
        issues = VerilogVerifier.check_syntax_patterns(code)
        
        if issues:
            print("\n⚠️  Potential Issues Found:")
            for i, issue in enumerate(issues, 1):
                print(f"  {i}. {issue}")
        else:
            print("\n✅ No obvious syntax issues detected")
        
        # Module information
        info = VerilogVerifier.extract_module_info(code)
        
        print("\n📋 Module Information:")
        for key, value in info.items():
            print(f"  • {key.replace('_', ' ').title()}: {value}")
        
        # Design style analysis
        print("\n🎨 Design Style:")
        if info.get('num_assigns', 0) > 0 and info.get('num_always_blocks', 0) == 0:
            print("  • Combinational logic using continuous assignments")
        elif info.get('num_always_blocks', 0) > 0 and not info.get('has_registers', False):
            print("  • Combinational logic using always blocks")
        elif info.get('num_always_blocks', 0) > 0 and info.get('has_registers', False):
            print("  • Sequential logic with registers")
        
        print("="*70)
        
        return len(issues) == 0

print("✓ Verification framework created!")

In [ ]:
# Verify some generated code
if MODEL_LOADED:
    print("Testing verification on a simple circuit...\n")
    
    test_prompt = """// Create a full adder
module full_adder(input a, input b, input cin, output sum, output cout);"""
    
    generated = generate_verilog(test_prompt, max_length=200, temperature=0.2)
    code = generated[0]
    
    display_generated_code(code)
    VerilogVerifier.verify_code(code)
else:
    print("⚠ Model not loaded - skipping verification demo")

## 11. Comparative Analysis Framework

Let's create tools to compare generated code against reference implementations.

In [ ]:
def compare_implementations(generated_code, reference_code, description=""):
    """
    Compare generated code with reference implementation
    """
    print(f"\n📊 COMPARISON ANALYSIS{': ' + description if description else ''}")
    print("="*70)
    
    # Extract info from both
    gen_info = VerilogVerifier.extract_module_info(generated_code)
    ref_info = VerilogVerifier.extract_module_info(reference_code)
    
    # Line counts
    gen_lines = len([l for l in generated_code.split('\n') if l.strip()])
    ref_lines = len([l for l in reference_code.split('\n') if l.strip()])
    
    print("\n📏 Code Metrics:")
    print(f"  {'Metric':<25} {'Generated':<15} {'Reference':<15} {'Match':<10}")
    print("-" * 70)
    print(f"  {'Total Lines':<25} {gen_lines:<15} {ref_lines:<15} {'✓' if gen_lines == ref_lines else '✗'}")
    print(f"  {'Number of Inputs':<25} {gen_info.get('num_inputs', 0):<15} {ref_info.get('num_inputs', 0):<15} "
          f"{'✓' if gen_info.get('num_inputs') == ref_info.get('num_inputs') else '✗'}")
    print(f"  {'Number of Outputs':<25} {gen_info.get('num_outputs', 0):<15} {ref_info.get('num_outputs', 0):<15} "
          f"{'✓' if gen_info.get('num_outputs') == ref_info.get('num_outputs') else '✗'}")
    print(f"  {'Always Blocks':<25} {gen_info.get('num_always_blocks', 0):<15} {ref_info.get('num_always_blocks', 0):<15} "
          f"{'✓' if gen_info.get('num_always_blocks') == ref_info.get('num_always_blocks') else '✗'}")
    print(f"  {'Assign Statements':<25} {gen_info.get('num_assigns', 0):<15} {ref_info.get('num_assigns', 0):<15} "
          f"{'✓' if gen_info.get('num_assigns') == ref_info.get('num_assigns') else '✗'}")
    
    # Check if approaches match
    print("\n🎨 Design Approach:")
    gen_style = "behavioral" if gen_info.get('num_always_blocks', 0) > 0 else "dataflow"
    ref_style = "behavioral" if ref_info.get('num_always_blocks', 0) > 0 else "dataflow"
    
    print(f"  Generated: {gen_style}")
    print(f"  Reference: {ref_style}")
    print(f"  Match: {'✓' if gen_style == ref_style else '✗ (different but both valid)'}")
    
    print("="*70)

print("✓ Comparison tools created!")

In [ ]:
# Run comparison on a benchmark
if MODEL_LOADED:
    print("Comparing generated code with reference implementation...\n")
    
    # Load a prompt from benchmarks
    with open(f'{benchmarks_dir}/basic2/prompt1_andgate.v', 'r') as f:
        prompt = f.read()
    
    # Load reference answer
    with open(f'{benchmarks_dir}/basic2/answer_andgate.v', 'r') as f:
        reference = f.read()
    
    # Generate code
    generated = generate_verilog(prompt, max_length=150, temperature=0.2)
    
    # Display both
    display_generated_code(generated[0], "🤖 Generated Code")
    display_generated_code(reference, "✅ Reference Code")
    
    # Compare
    compare_implementations(generated[0], reference, "AND Gate")
else:
    print("⚠ Model not loaded - skipping comparison")

## 12. Batch Evaluation Across Benchmarks

Let's evaluate the model on multiple benchmarks systematically.

In [ ]:
import pandas as pd

def evaluate_benchmarks(benchmark_list, max_per_category=3):
    """
    Evaluate model on a list of benchmarks
    """
    results = []
    
    print("🔬 BATCH EVALUATION")
    print("="*70)
    
    for benchmark in benchmark_list:
        bench_path = os.path.join(benchmarks_dir, benchmark)
        if not os.path.exists(bench_path):
            continue
        
        # Find prompt and answer files
        prompt_files = sorted([f for f in os.listdir(bench_path) if f.startswith('prompt')])
        answer_files = sorted([f for f in os.listdir(bench_path) if f.startswith('answer')])
        
        if not prompt_files or not answer_files:
            continue
        
        print(f"\n📝 Evaluating: {benchmark}")
        print("-" * 70)
        
        # Use first prompt
        with open(os.path.join(bench_path, prompt_files[0]), 'r') as f:
            prompt = f.read()
        
        with open(os.path.join(bench_path, answer_files[0]), 'r') as f:
            reference = f.read()
        
        # Generate code
        try:
            generated = generate_verilog(prompt, max_length=300, temperature=0.2)
            code = generated[0]
            
            # Verify
            issues = VerilogVerifier.check_syntax_patterns(code)
            syntax_ok = len(issues) == 0
            
            # Extract info
            gen_info = VerilogVerifier.extract_module_info(code)
            ref_info = VerilogVerifier.extract_module_info(reference)
            
            # Check if I/O matches
            io_match = (gen_info.get('num_inputs') == ref_info.get('num_inputs') and 
                       gen_info.get('num_outputs') == ref_info.get('num_outputs'))
            
            result = {
                'Benchmark': benchmark,
                'Syntax OK': '✓' if syntax_ok else '✗',
                'I/O Match': '✓' if io_match else '✗',
                'Lines': len([l for l in code.split('\n') if l.strip()]),
                'Style': 'behavioral' if gen_info.get('num_always_blocks', 0) > 0 else 'dataflow'
            }
            
            results.append(result)
            print(f"  Syntax: {'✓' if syntax_ok else '✗'}  |  I/O Match: {'✓' if io_match else '✗'}")
            
        except Exception as e:
            print(f"  ❌ Error: {str(e)[:50]}")
            results.append({
                'Benchmark': benchmark,
                'Syntax OK': '❌',
                'I/O Match': '❌',
                'Lines': 0,
                'Style': 'error'
            })
    
    return pd.DataFrame(results)

if MODEL_LOADED:
    # Select a subset of benchmarks
    test_benchmarks = [
        'basic1', 'basic2', 'basic3',
        'intermediate1', 'intermediate2', 'intermediate3',
        'advanced1'
    ]
    
    results_df = evaluate_benchmarks(test_benchmarks)
    
    print("\n\n📊 EVALUATION SUMMARY")
    print("="*70)
    print(results_df.to_string(index=False))
    
    # Calculate success rates
    syntax_success = (results_df['Syntax OK'] == '✓').sum()
    io_success = (results_df['I/O Match'] == '✓').sum()
    total = len(results_df)
    
    print(f"\n✅ Success Rates:")
    print(f"  Syntax Correctness: {syntax_success}/{total} ({100*syntax_success/total:.1f}%)")
    print(f"  I/O Match: {io_success}/{total} ({100*io_success/total:.1f}%)")
else:
    print("⚠ Model not loaded - skipping batch evaluation")

## 13. Interactive Playground

Try your own prompts and experiment with the model!

In [ ]:
# Interactive generation function
def generate_from_description(description, 
                              module_name="custom_module",
                              show_analysis=True):
    """
    Generate Verilog from a natural language description
    """
    # Create a prompt from the description
    prompt = f"// {description}\nmodule {module_name}"
    
    print(f"\n🎯 Task: {description}")
    print("="*70)
    print(f"\n📝 Generated Prompt:\n{prompt}")
    
    if MODEL_LOADED:
        generated = generate_verilog(prompt, max_length=300, temperature=0.3)
        code = generated[0]
        
        display_generated_code(code)
        
        if show_analysis:
            analyze_code(code)
            VerilogVerifier.verify_code(code)
        
        return code
    else:
        print("\n⚠ Model not loaded")
        return None

# Example usage
print("🎮 INTERACTIVE PLAYGROUND")
print("="*70)
print("Try these examples or create your own!\n")

# Example 1
generate_from_description(
    "Create a 4-bit binary counter with asynchronous reset",
    module_name="counter_4bit"
)

In [ ]:
# Try your own custom prompt here!
# Uncomment and modify the description below:

# generate_from_description(
#     "Your description here",
#     module_name="your_module_name"
# )

print("Uncomment the code above and add your own description to try it!")
print("\nSuggested ideas:")
print("  • 'Design a 3-to-8 decoder'")
print("  • 'Create a priority encoder with 8 inputs'")
print("  • 'Implement a simple UART transmitter'")
print("  • 'Build a BCD to 7-segment decoder'")
print("  • 'Design a traffic light controller FSM'")

## 14. Student Exercises and Challenges

Test your understanding with these hands-on exercises!

### 📝 Exercise 1: Prompt Engineering (Beginner)

**Task**: Create 3 different prompts for generating a 2-to-4 decoder. Compare the outputs.

**Questions**:
1. Which prompt style produces the most accurate code?
2. How does prompt phrasing affect the implementation style (behavioral vs. dataflow)?
3. What happens if you add more context in comments?

---

### 🔧 Exercise 2: Parameter Tuning (Intermediate)

**Task**: Generate a 4-bit ripple carry adder with different temperature values (0.1, 0.5, 1.0).

**Questions**:
1. How does temperature affect code correctness?
2. At what temperature do you get the best balance of creativity and correctness?
3. When might higher temperature be beneficial?

---

### 🎯 Exercise 3: Comparative Analysis (Intermediate)

**Task**: Generate code for benchmarks at all three difficulty levels. Create a table comparing:
- Syntax correctness
- I/O matching with reference
- Code length
- Implementation style

**Questions**:
1. Does model performance degrade with problem complexity?
2. Which types of modules are hardest for the model?
3. What patterns do you notice in successful vs. failed generations?

---

### 🚀 Exercise 4: Design Challenge (Advanced)

**Task**: Create a complex design not in the benchmarks (e.g., FIFO buffer, SPI controller, or simple CPU component).

**Requirements**:
1. Write a detailed prompt
2. Generate multiple versions
3. Manually verify the best version
4. Write a simple testbench
5. Document any issues found

---

### 🔬 Exercise 5: Research Extension (Advanced)

**Task**: Compare VGen's fine-tuned model with a base model (e.g., regular CodeGen or GPT).

**Questions**:
1. How much does fine-tuning improve Verilog generation?
2. What types of errors does fine-tuning eliminate?
3. Are there cases where the base model performs better?


In [ ]:
# Exercise Workspace
# Use this cell to complete the exercises above

print("Exercise Workspace")
print("="*70)
print("Use this cell to experiment with the exercises above.")
print("\nExample for Exercise 1:")
print("""
# Prompt 1: Direct specification
prompt1 = "// 2-to-4 decoder\nmodule decoder_2to4("

# Prompt 2: Detailed description
prompt2 = "// Create a 2-to-4 decoder with enable\nmodule decoder_2to4("

# Prompt 3: With truth table
prompt3 = "// 2-to-4 decoder: when enable=1, output[i]=1 where i matches input\nmodule decoder_2to4("

# Generate and compare
# ...
""")

# Your code here:

## 15. Best Practices and Guidelines

### ✅ Effective Prompt Engineering for Verilog Generation

#### 1. **Be Specific and Detailed**
```verilog
// ❌ Bad: Too vague
// Create a counter

// ✅ Good: Specific requirements
// Create a 4-bit up counter with synchronous reset
// Count from 0 to 15, then wrap to 0
// Reset sets count to 0
```

#### 2. **Include Port Specifications**
```verilog
// ✅ Provide port information
module counter(
    input clk,
    input reset,
    output reg [3:0] count
);
```

#### 3. **Specify Behavior Clearly**
```verilog
// ✅ Describe the behavior
// On rising edge of clk:
//   if reset is high, count = 0
//   else count = count + 1
```

#### 4. **Use Consistent Naming**
- Use standard signal names (clk, reset, enable)
- Follow common conventions (active high vs. active low)
- Be consistent with naming style

#### 5. **Start Simple, Add Complexity**
- Generate basic functionality first
- Add features incrementally
- Verify each step

### ⚠️ Common Pitfalls to Avoid

1. **Ambiguous Specifications**
   - Unclear edge sensitivity (posedge vs negedge)
   - Unspecified reset behavior
   - Missing signal widths

2. **Over-constraining**
   - Too many specific implementation details
   - Restricting the model unnecessarily

3. **Under-constraining**
   - Too vague descriptions
   - Missing critical functionality

### 🔍 Verification Checklist

Before using generated code:
- [ ] Check module/endmodule pairing
- [ ] Verify all ports are declared
- [ ] Ensure balanced begin/end blocks
- [ ] Check for proper sensitivity lists in always blocks
- [ ] Verify reset logic is correct
- [ ] Look for unintended latches (missing else clauses)
- [ ] Confirm signal widths match requirements
- [ ] Test with a proper testbench

### 🎯 When to Use LLM-Generated Verilog

**Good Use Cases**:
- ✅ Prototyping simple modules
- ✅ Learning Verilog syntax and patterns
- ✅ Generating testbench templates
- ✅ Quick exploration of design alternatives
- ✅ Boilerplate code generation

**Use with Caution**:
- ⚠️ Complex state machines
- ⚠️ Timing-critical designs
- ⚠️ Safety-critical applications
- ⚠️ Production code without thorough verification

### 📚 Learning Resources

- **Verilog Reference**: IEEE 1364-2005 Standard
- **Best Practices**: Sutherland's Verilog and SystemVerilog Gotchas
- **Synthesis**: Synopsys Design Compiler documentation
- **Verification**: SystemVerilog for Verification

## 16. Understanding Model Limitations

### 🚧 Current Limitations of LLM-based Verilog Generation

#### 1. **Functional Correctness**
- May generate syntactically correct but functionally incorrect code
- Can miss edge cases and corner cases
- May not handle all reset/initialization scenarios

#### 2. **Complex Designs**
- Struggles with multi-module hierarchies
- May not maintain consistency across related modules
- Limited understanding of system-level architecture

#### 3. **Timing and Performance**
- No awareness of timing constraints
- May not optimize for synthesis
- Cannot reason about clock domain crossings

#### 4. **Verification**
- Cannot fully verify its own output
- May not cover all test scenarios
- Limited formal verification capabilities

### 🔮 Future Directions

1. **Improved Fine-tuning**
   - Larger, more diverse Verilog datasets
   - Reinforcement learning from synthesis/simulation feedback
   - Multi-task learning with verification

2. **Tool Integration**
   - Direct integration with EDA tools
   - Automated synthesis and verification loops
   - Real-time feedback during generation

3. **Enhanced Capabilities**
   - Better understanding of hierarchical designs
   - Awareness of timing and physical constraints
   - Automated testbench generation

4. **Specialized Models**
   - Domain-specific models for different types of circuits
   - Models trained on verified IP libraries
   - Integration with formal verification tools

## 17. Summary and Conclusions

### 🎓 What We've Learned

In this comprehensive tutorial, we explored:

1. **LLM Fundamentals for Hardware Design**
   - How transformer models can understand Verilog syntax
   - The importance of domain-specific fine-tuning
   - Trade-offs between model size and performance

2. **Practical Verilog Generation**
   - Generating basic combinational circuits
   - Creating sequential logic and state machines
   - Tackling advanced designs with complex requirements

3. **Evaluation and Verification**
   - Syntax checking and pattern matching
   - Comparison with reference implementations
   - Batch evaluation across benchmarks

4. **Best Practices**
   - Effective prompt engineering techniques
   - Parameter tuning for better results
   - Understanding when to use (and not use) LLM-generated code

### 📊 Key Findings from Research

From the VGen paper (Thakur et al., 2022):

- **Fine-tuning improves syntax**: 25.9% overall improvement in syntactic correctness
- **Functional correctness**: Fine-tuned CodeGen outperforms commercial Codex by 6.5%
- **Problem complexity matters**: Performance degrades on advanced benchmarks
- **Model size helps**: Larger models generally perform better

### 🚀 The Future of AI-Assisted Hardware Design

**Near-term (1-2 years)**:
- Better code completion in hardware IDEs
- Automated testbench generation
- Natural language to RTL prototyping

**Medium-term (3-5 years)**:
- End-to-end design from specifications
- Automated optimization and refactoring
- Integration with formal verification

**Long-term (5+ years)**:
- AI-driven architecture exploration
- Automated debugging and error correction
- Collaborative human-AI design workflows

### 💡 Key Takeaways

1. **LLMs are powerful assistants, not replacements**
   - Use for prototyping and learning
   - Always verify generated code
   - Understand the underlying hardware concepts

2. **Quality depends on prompts**
   - Clear, detailed specifications yield better results
   - Iterative refinement improves output
   - Domain knowledge helps craft better prompts

3. **Verification is essential**
   - Never trust generated code blindly
   - Use proper testbenches
   - Understand what the code does

4. **The field is rapidly evolving**
   - Models are improving quickly
   - New techniques emerge regularly
   - Stay updated with latest research

### 📚 Further Reading

**Research Papers**:
- VGen paper: [arXiv:2212.11140](https://arxiv.org/abs/2212.11140)
- CodeGen: [arXiv:2203.13474](https://arxiv.org/abs/2203.13474)
- Related work in code generation and hardware synthesis

**Resources**:
- VGen GitHub: https://github.com/shailja-thakur/VGen
- HuggingFace Models: https://huggingface.co/shailja
- Verilog tutorials and references

### 🙏 Acknowledgments

This tutorial is based on the VGen framework developed by:
- Shailja Thakur, Baleegh Ahmad, Zhenxing Fan, Hammond Pearce
- Benjamin Tan, Ramesh Karri, Brendan Dolan-Gavitt, Siddharth Garg
- NYU Tandon School of Engineering

### 📧 Contact and Feedback

For questions about this tutorial or VGen:
- GitHub Issues: https://github.com/shailja-thakur/VGen/issues
- Research Group: NYU CSAW Hardware Security

---

## Thank you for completing this tutorial! 🎉

We hope you've gained valuable insights into using LLMs for hardware design. 
Keep exploring, experimenting, and pushing the boundaries of AI-assisted chip design!

In [ ]:
# Final statistics and summary
print("\n" + "="*70)
print(" "*20 + "TUTORIAL COMPLETE!" + " "*20)
print("="*70)
print("\n🎓 You've completed the VGen tutorial for LLM4ChipDesign!")
print("\n📊 Session Statistics:")
print(f"  • Model Loaded: {'Yes ✓' if MODEL_LOADED else 'No ✗'}")
print(f"  • Device Used: {device}")
print(f"  • Examples Covered: 17 sections")
print("\n💡 Key Skills Acquired:")
print("  ✓ Understanding LLM-based Verilog generation")
print("  ✓ Effective prompt engineering for hardware design")
print("  ✓ Code verification and evaluation techniques")
print("  ✓ Comparative analysis of generated vs. reference code")
print("  ✓ Best practices for AI-assisted chip design")
print("\n🚀 Next Steps:")
print("  1. Try the exercises with your own designs")
print("  2. Explore the full VGen benchmark suite")
print("  3. Read the research paper for deeper insights")
print("  4. Contribute to the VGen project on GitHub")
print("\n📚 Remember: AI is a powerful tool, but human expertise is irreplaceable!")
print("\n" + "="*70)
print("\nThank you for your participation! 🙏")
print("\nFor questions or feedback, visit:")
print("  https://github.com/shailja-thakur/VGen")
print("\n" + "="*70)